In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip -q install opencv-python numpy

In [3]:
import os
import math
import cv2
import numpy as np
from pathlib import Path

In [4]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 7.4 MB/s eta 0:00:00


In [5]:
import cv2
import matplotlib.pyplot as plt

cap = cv2.VideoCapture(input_video_path)
ret, frame = cap.read()
cap.release()

rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(18,4))
plt.imshow(rgb)
plt.xlim(0, rgb.shape[1])
plt.ylim(rgb.shape[0], 0)
plt.grid()
plt.show()

NameError: name 'input_video_path' is not defined

In [6]:
import os
import cv2
import numpy as np
from pathlib import Path
from ultralytics import YOLO

# 1) (Colab) mount drive if needed
from google.colab import drive
drive.mount('/content/drive')

# -----------------------------
# Config (EDIT THESE)
# -----------------------------
input_video_path = "/content/drive/MyDrive/CREATE Lab/video_agent360/output_top.mp4"

out_dir = "/content/drive/MyDrive/CREATE Lab/video_agent360/autocrop_out"
Path(out_dir).mkdir(parents=True, exist_ok=True)

# Your stitched layout assumption:
# top strip height = TOP_H
# below is the area containing participant regions
TOP_H = 0
LOW_Y0 = 0
LOW_H  = None   # None -> infer as (H - TOP_H)

# Per-person output crop size
OUT_W = 640
OUT_H = 900

# Sampling for auto bbox estimation
SAMPLE_EVERY_N_FRAMES = 10
MAX_SAMPLE_FRAMES = 300

# YOLO model
YOLO_MODEL = "yolov8n.pt"

# Crop design knobs
MARGIN_X = 0.55
MARGIN_TOP = 0.25
MARGIN_BOTTOM = 1.10
DOWN_SHIFT = 0.20

# If None -> auto split into 4 equal columns
# You can also manually set this, e.g.
# REGION_X_BOUNDS = [(0,420), (420,960), (960,1500), (1500,1920)]
REGION_X_BOUNDS = None

# Output fps (None = keep original)
OUT_FPS = None

# Detection settings
YOLO_CONF = 0.25
YOLO_IOU = 0.45
MIN_BOX_W = 40
MIN_BOX_H = 60


# -----------------------------
# Helpers
# -----------------------------
def clamp(v, lo, hi):
    return max(lo, min(hi, v))

def compute_crop_window_from_bbox(
    bbox_xyxy, frame_w, frame_h,
    out_w=640, out_h=900,
    margin_x=0.55, margin_top=0.25, margin_bottom=1.10,
    down_shift=0.20
):
    """
    bbox_xyxy: [x1,y1,x2,y2] for person
    returns crop window (x1,y1,x2,y2) in frame coords, size close to out_w:out_h
    """
    x1, y1, x2, y2 = bbox_xyxy
    bw = x2 - x1
    bh = y2 - y1
    cx = (x1 + x2) / 2.0
    cy = (y1 + y2) / 2.0 + down_shift * bh  # bias down to include desk

    # expand bbox asymmetric
    ex1 = x1 - margin_x * bw
    ex2 = x2 + margin_x * bw
    ey1 = y1 - margin_top * bh
    ey2 = y2 + margin_bottom * bh

    target_ar = out_w / out_h

    w = (ex2 - ex1)
    h = (ey2 - ey1)

    cur_ar = w / h if h > 1e-6 else target_ar
    if cur_ar > target_ar:
        h = w / target_ar
    else:
        w = h * target_ar

    crop_x1 = cx - w / 2.0
    crop_x2 = cx + w / 2.0
    crop_y1 = cy - h / 2.0
    crop_y2 = cy + h / 2.0

    crop_x1 = clamp(crop_x1, 0, frame_w - 2)
    crop_y1 = clamp(crop_y1, 0, frame_h - 2)
    crop_x2 = clamp(crop_x2, crop_x1 + 1, frame_w - 1)
    crop_y2 = clamp(crop_y2, crop_y1 + 1, frame_h - 1)

    return [int(crop_x1), int(crop_y1), int(crop_x2), int(crop_y2)]

def assign_to_slot(x_center, bounds):
    for i, (a, b) in enumerate(bounds):
        if a <= x_center < b:
            return i
    mids = [(a + b) / 2 for a, b in bounds]
    return int(np.argmin([abs(x_center - m) for m in mids]))

def median_bbox(bboxes):
    arr = np.array(bboxes, dtype=np.float32)
    return np.median(arr, axis=0).tolist()


# -----------------------------
# 2) Pass 1: sample frames -> estimate stable bbox per slot
# -----------------------------
cap = cv2.VideoCapture(input_video_path)
if not cap.isOpened():
    raise RuntimeError(f"Cannot open video: {input_video_path}")

W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
in_fps = cap.get(cv2.CAP_PROP_FPS)
fps = OUT_FPS if OUT_FPS is not None else (in_fps if in_fps > 0 else 30.0)

if LOW_H is None:
    LOW_H = H - TOP_H

if LOW_H <= 0:
    raise RuntimeError(f"Invalid LOW_H={LOW_H}. Check TOP_H / LOW_Y0 / LOW_H.")

# slot bounds
if REGION_X_BOUNDS is None:
    step = W / 4.0
    REGION_X_BOUNDS = [(int(i * step), int((i + 1) * step)) for i in range(4)]

NUM_SLOTS = len(REGION_X_BOUNDS)

print("Frame size:", W, H, "TOP_H:", TOP_H, "LOW_Y0:", LOW_Y0, "LOW_H:", LOW_H)
print("Slot bounds:", REGION_X_BOUNDS)
print("NUM_SLOTS:", NUM_SLOTS)

model = YOLO(YOLO_MODEL)

slot_bboxes = [[] for _ in range(NUM_SLOTS)]
sampled = 0
frame_idx = 0

while True:
    ok, frame = cap.read()
    if not ok:
        break

    if frame_idx % SAMPLE_EVERY_N_FRAMES != 0:
        frame_idx += 1
        continue

    roi = frame[LOW_Y0:LOW_Y0 + LOW_H, :, :]
    if roi.size == 0:
        raise RuntimeError(
            f"Empty ROI. Check TOP_H/LOW_Y0/LOW_H. H={H}, LOW_Y0={LOW_Y0}, LOW_H={LOW_H}"
        )

    roi_rgb = cv2.cvtColor(roi, cv2.COLOR_BGR2RGB)

    results = model.predict(roi_rgb, conf=YOLO_CONF, iou=YOLO_IOU, verbose=False)
    det = results[0]

    if det.boxes is not None and len(det.boxes) > 0:
        for b, c in zip(det.boxes.xyxy.cpu().numpy(), det.boxes.cls.cpu().numpy()):
            if int(c) != 0:   # class 0 = person
                continue

            x1, y1, x2, y2 = b.tolist()
            x1f, y1f, x2f, y2f = x1, y1 + LOW_Y0, x2, y2 + LOW_Y0

            if (x2f - x1f) < MIN_BOX_W or (y2f - y1f) < MIN_BOX_H:
                continue

            xc = (x1f + x2f) / 2.0
            slot = assign_to_slot(xc, REGION_X_BOUNDS)
            slot_bboxes[slot].append([x1f, y1f, x2f, y2f])

    sampled += 1
    frame_idx += 1

    if sampled >= MAX_SAMPLE_FRAMES:
        break

cap.release()

print("\nCollected bbox counts:")
for i in range(NUM_SLOTS):
    print(f"slot {i} collected bboxes: {len(slot_bboxes[i])}")

# -----------------------------
# 3) Compute median bbox per slot (allow empty slots)
# -----------------------------
median_bboxes = [None] * NUM_SLOTS
active_slots = []

for i in range(NUM_SLOTS):
    if len(slot_bboxes[i]) == 0:
        print(f"Warning: slot {i} is empty, skipping.")
    else:
        median_bboxes[i] = median_bbox(slot_bboxes[i])
        active_slots.append(i)

if len(active_slots) == 0:
    raise RuntimeError(
        "No participant detections found in any slot. "
        "Try lowering YOLO_CONF, increasing MAX_SAMPLE_FRAMES, or checking layout."
    )

print("\nActive slots:", active_slots)
print("Median bboxes (xyxy):")
for i in active_slots:
    print(f"slot {i}:", [int(x) for x in median_bboxes[i]])

# -----------------------------
# 4) Build crop windows only for active slots
# -----------------------------
crop_windows = {}
for i in active_slots:
    bb = median_bboxes[i]
    cw = compute_crop_window_from_bbox(
        bb, frame_w=W, frame_h=H,
        out_w=OUT_W, out_h=OUT_H,
        margin_x=MARGIN_X,
        margin_top=MARGIN_TOP,
        margin_bottom=MARGIN_BOTTOM,
        down_shift=DOWN_SHIFT
    )
    crop_windows[i] = cw

print("\nCrop windows (x1,y1,x2,y2):")
for i in active_slots:
    cw = crop_windows[i]
    print(f"slot {i}:", cw, " size=", (cw[2] - cw[0], cw[3] - cw[1]))

# -----------------------------
# 5) Pass 2: crop & export only active slots
# -----------------------------
cap = cv2.VideoCapture(input_video_path)
if not cap.isOpened():
    raise RuntimeError(f"Cannot open video: {input_video_path}")

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writers = {}
out_paths = {}

for i in active_slots:
    out_path = str(Path(out_dir) / f"person_slot{i}_{OUT_W}x{OUT_H}.mp4")
    w = cv2.VideoWriter(out_path, fourcc, fps, (OUT_W, OUT_H))
    if not w.isOpened():
        raise RuntimeError(f"Cannot open writer: {out_path}")
    writers[i] = w
    out_paths[i] = out_path

frame_idx = 0
while True:
    ok, frame = cap.read()
    if not ok:
        break

    for i in active_slots:
        x1, y1, x2, y2 = crop_windows[i]
        crop = frame[y1:y2, x1:x2, :]
        if crop.size == 0:
            print(f"Warning: empty crop on frame {frame_idx}, slot {i}")
            continue
        crop_resized = cv2.resize(crop, (OUT_W, OUT_H), interpolation=cv2.INTER_LINEAR)
        writers[i].write(crop_resized)

    frame_idx += 1
    if frame_idx % 200 == 0:
        print("exported frames:", frame_idx)

cap.release()
for i in active_slots:
    writers[i].release()

print("\nDone. Outputs:")
for i in active_slots:
    print(f" - slot {i}: {out_paths[i]}")

# -----------------------------
# 6) Quick sanity check: save one debug frame with crop boxes drawn
# -----------------------------
cap = cv2.VideoCapture(input_video_path)
ok, frame = cap.read()
cap.release()

if ok:
    dbg = frame.copy()

    # draw slot regions
    for i, (sx0, sx1) in enumerate(REGION_X_BOUNDS):
        color = (255, 255, 0)
        cv2.rectangle(dbg, (sx0, 5), (sx1, 40), color, 2)
        cv2.putText(dbg, f"slot {i}", (sx0 + 10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

    # draw crop windows only for active slots
    for i in active_slots:
        x1, y1, x2, y2 = crop_windows[i]
        cv2.rectangle(dbg, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(dbg, f"P_slot{i}", (x1 + 10, y1 + 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 2)

    dbg_path = str(Path(out_dir) / "debug_crop_windows.jpg")
    cv2.imwrite(dbg_path, dbg)
    print("Saved debug image:", dbg_path)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Frame size: 1920 360 TOP_H: 0 LOW_Y0: 0 LOW_H: 360
Slot bounds: [(0, 480), (480, 960), (960, 1440), (1440, 1920)]
NUM_SLOTS: 4

Collected bbox counts:
slot 0 collected bboxes: 312
slot 1 collected bboxes: 0
slot 2 collected bboxes: 293
slot 3 collected bboxes: 348

Active slots: [0, 2, 3]
Median bboxes (xyxy):
slot 0: [159, 122, 363, 348]
slot 2: [1076, 185, 1257, 346]
slot 3: [1527, 145, 1784, 352]

Crop windows (x1,y1,x2,y2):
slot 0: [47, 0, 475, 359]  size= (428, 359)
slot 2: [976, 29, 1357, 359]  size= (381, 330)
slot 3: [1386, 0, 1919, 359]  size= (